In [1]:
import os, sys

os.chdir(os.path.expanduser("~/QIAO0042/models/acv/facemask/"))
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())


CWD: /scratch-share/QIAO0042/models/acv/facemask


In [2]:
import os
import time
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import wandb

from unet import UNetV2
from losses import FocalDiceLoss
from metrics import confusion_matrix, f1_macro_from_cm
from palette import NUM_CLASSES
from dataset import FaceParsingDataset
from augment import make_face_aug


In [3]:
import shutil, time as _t
from pathlib import Path

def copy_to_tmp(src_dirs, tmp_root="/tmp/facemask"):
    mapping = {}
    for src in src_dirs:
        src_path = Path(src).resolve()
        dst_path = Path(tmp_root) / src_path.name
        if dst_path.exists():
            print(f"  /tmp cache already exists: {dst_path}")
        else:
            t0 = _t.time()
            shutil.copytree(src_path, dst_path)
            print(f"  copied {src_path} → {dst_path}  ({_t.time()-t0:.1f}s)")
        mapping[str(src)] = str(dst_path)
    return mapping

tmp_map   = copy_to_tmp(["train/images", "train/masks"])
_img_dir  = tmp_map["train/images"]
_mask_dir = tmp_map["train/masks"]
print("Data dirs:", _img_dir, _mask_dir)

aug_fn = make_face_aug(p_flip=0.5, p_geom=0.7, p_color=0.7, p_blur=0.15)


  /tmp cache already exists: /tmp/facemask/images
  /tmp cache already exists: /tmp/facemask/masks
Data dirs: /tmp/facemask/images /tmp/facemask/masks


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/scratch-share/QIAO0042/models/acv/facemask/augment.py:166: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(


In [4]:
import numpy as np
import logging
from pathlib import Path
from PIL import Image as PILImage
from palette import rgb_to_label

logging.getLogger("torch._inductor").setLevel(logging.WARNING)
logging.getLogger("torch._dynamo").setLevel(logging.WARNING)

img_dir  = _img_dir
mask_dir = _mask_dir

batch_size      = 16
lr              = 2e-2
warmup_epochs   = 5
epochs          = 100
base_width      = 23
dropout         = 0.3
aux_weight      = 0.4
focal_gamma     = 2.0
dice_w          = 0.85
ohem_ratio      = 0.7
ema_decay       = 0.999
ckpt_path       = "checkpoints/full_best.pt"
device          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

use_bf16  = device.type == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"AMP dtype : {amp_dtype}")

torch.backends.cudnn.benchmark = True

# --- class weights (sqrt median-frequency balancing over all 1000 masks) ---
print("Computing class pixel frequencies ...")
pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
mask_lookup  = {
    p.stem: p for p in Path(mask_dir).iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
}
for mask_path in mask_lookup.values():
    mask_rgb = np.array(PILImage.open(mask_path).convert("RGB"), dtype=np.uint8)
    labels   = rgb_to_label(mask_rgb)
    for c in range(NUM_CLASSES):
        pixel_counts[c] += int((labels == c).sum())

freq             = pixel_counts / pixel_counts.sum()
median_freq      = float(np.median(freq[freq > 0]))
class_weights_np = np.where(freq > 0, np.sqrt(median_freq / freq), 1.0)
class_weights    = torch.tensor(class_weights_np, dtype=torch.float32)

print(f"{'cls':>4}  {'freq':>8}  {'weight':>8}")
for i, (f, w) in enumerate(zip(freq, class_weights_np)):
    print(f"  {i:2d}   {f:.5f}   {w:.3f}")

# --- dataset: all 1000 labelled images for training ---
train_ds = FaceParsingDataset(img_dir, mask_dir, file_list=None,
                              augment=aug_fn, cache=True)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=8, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

# --- model ---
model = UNetV2(num_classes=NUM_CLASSES, base=base_width,
               dropout=dropout, deep_supervision=True).to(device)
try:
    model = torch.compile(model, mode="default")
    print("torch.compile: enabled")
except Exception as e:
    print(f"torch.compile: skipped ({e})")

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- optimizer + scheduler ---
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6),
    ],
    milestones=[warmup_epochs],
)

# --- loss ---
criterion = FocalDiceLoss(
    num_classes=NUM_CLASSES,
    dice_weight=dice_w,
    gamma=focal_gamma,
    class_weights=class_weights.to(device),
    ohem_ratio=ohem_ratio,
)

scaler = torch.amp.GradScaler("cuda", enabled=(not use_bf16 and device.type == "cuda"))

steps_per_epoch = len(train_loader)
print(f"\nModel     : UNetV2(base={base_width})  |  Params: {num_params:,}")
print(f"Device    : {device}  |  AMP: {amp_dtype}  |  cudnn.benchmark: ON")
print(f"Training  : {epochs} epochs × {steps_per_epoch} steps  (batch={batch_size}, all 1000 images)")
print(f"Loss      : FocalDice(gamma={focal_gamma}, dice_w={dice_w}, OHEM={ohem_ratio})")
print(f"EMA       : decay={ema_decay} (with warmup)")

wandb.init(
    project="face-parsing-unet",
    name=f"unetv2_full1000_ohem{ohem_ratio}_L40S",
    config={
        "model":            "UNetV2",
        "num_classes":      NUM_CLASSES,
        "base_width":       base_width,
        "dropout":          dropout,
        "params":           num_params,
        "loss":             "FocalDice+OHEM",
        "focal_gamma":      focal_gamma,
        "dice_weight":      dice_w,
        "ohem_ratio":       ohem_ratio,
        "aux_weight":       aux_weight,
        "class_weights":    "sqrt(median/freq)",
        "lr":               lr,
        "warmup_epochs":    warmup_epochs,
        "scheduler":        "LinearWarmup+CosineAnnealingLR",
        "batch_size":       batch_size,
        "epochs":           epochs,
        "train_samples":    len(train_ds),
        "ema_decay":        ema_decay,
        "amp_dtype":        str(amp_dtype),
        "augmentation":     "hflip+label_swap, ShiftScaleRotate, ColorJitter, GaussianBlur",
        "deep_supervision": True,
        "gpu":              "L40S",
    },
)


AMP dtype : torch.bfloat16
Computing class pixel frequencies ...
 cls      freq    weight
   0   0.28972   0.120
   1   0.25231   0.129
   2   0.02062   0.451
   3   0.00266   1.255
   4   0.00223   1.370
   5   0.00221   1.376
   6   0.00418   1.001
   7   0.00408   1.014
   8   0.00466   0.948
   9   0.00383   1.047
  10   0.00322   1.142
  11   0.00419   1.000
  12   0.00683   0.784
  13   0.31525   0.115
  14   0.00665   0.794
  15   0.00254   1.284
  16   0.00017   5.005
  17   0.04012   0.323
  18   0.03454   0.348
  → cached 1000 decoded arrays in RAM
Loaded 1000 samples from /tmp/facemask/images


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/msai/qiao0042/.netrc.


torch.compile: enabled

Model     : UNetV2(base=23)  |  Params: 1,796,664
Device    : cuda  |  AMP: torch.bfloat16  |  cudnn.benchmark: ON
Training  : 100 epochs × 63 steps  (batch=16, all 1000 images)
Loss      : FocalDice(gamma=2.0, dice_w=0.85, OHEM=0.7)
EMA       : decay=0.999 (with warmup)


wandb: Currently logged in as: 584832452 (584832452-nanyang-technological-university-singapore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
from contextlib import contextmanager

class EMAKeeper:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.num_updates = 0
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        decay = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(decay).add_(p.data, alpha=1 - decay)


def _downsample_mask(masks, size):
    return F.interpolate(
        masks.float().unsqueeze(1), size=size, mode="nearest"
    ).squeeze(1).long()


def train():
    print(f"Trainable params: {num_params:,}")
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    ema = EMAKeeper(model, decay=ema_decay)

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        running = 0.0

        for imgs, masks, _ in train_loader:
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                outputs = model(imgs)
                if isinstance(outputs, tuple):
                    main, aux1, aux2 = outputs
                    loss = (criterion(main, masks)
                            + aux_weight * criterion(aux1, _downsample_mask(masks, aux1.shape[-2:]))
                            + aux_weight * criterion(aux2, _downsample_mask(masks, aux2.shape[-2:])))
                else:
                    loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)
            running += loss.item()

        train_loss = running / max(1, len(train_loader))
        scheduler.step()
        elapsed = time.time() - t0

        wandb.log({
            "epoch":          epoch,
            "train/loss":     train_loss,
            "time/epoch_sec": elapsed,
            "lr":             scheduler.get_last_lr()[0],
        })
        print(f"[{epoch:03d}/{epochs}] loss={train_loss:.4f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}  time={elapsed:.1f}s")

    # Save final checkpoint with EMA shadow for inference
    torch.save({
        "model":          model.state_dict(),
        "ema_shadow":     ema.shadow,
        "epoch":          epochs,
        "config":         dict(wandb.config),
    }, ckpt_path)
    print(f"\nSaved {ckpt_path}")
    wandb.finish()


In [7]:
train()


Trainable params: 1,796,664
[001/100] loss=1.9968  lr=5.60e-03  time=30.2s
[002/100] loss=1.3372  lr=9.20e-03  time=9.2s
[003/100] loss=1.0450  lr=1.28e-02  time=9.0s
[004/100] loss=0.9717  lr=1.64e-02  time=9.1s


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[005/100] loss=0.9043  lr=2.00e-02  time=9.1s
[006/100] loss=0.8641  lr=2.00e-02  time=9.3s
[007/100] loss=0.8079  lr=2.00e-02  time=9.1s
[008/100] loss=0.7514  lr=2.00e-02  time=9.1s
[009/100] loss=0.7312  lr=1.99e-02  time=9.1s
[010/100] loss=0.7099  lr=1.99e-02  time=9.0s
[011/100] loss=0.7059  lr=1.98e-02  time=9.3s
[012/100] loss=0.6881  lr=1.97e-02  time=9.3s
[013/100] loss=0.6611  lr=1.97e-02  time=9.1s
[014/100] loss=0.6410  lr=1.96e-02  time=9.2s
[015/100] loss=0.6445  lr=1.95e-02  time=9.0s
[016/100] loss=0.6492  lr=1.93e-02  time=9.2s
[017/100] loss=0.6577  lr=1.92e-02  time=9.2s
[018/100] loss=0.6113  lr=1.91e-02  time=9.1s
[019/100] loss=0.6046  lr=1.89e-02  time=9.0s
[020/100] loss=0.6061  lr=1.88e-02  time=9.3s
[021/100] loss=0.5744  lr=1.86e-02  time=9.0s
[022/100] loss=0.5979  lr=1.85e-02  time=9.2s
[023/100] loss=0.5727  lr=1.83e-02  time=9.2s
[024/100] loss=0.5530  lr=1.81e-02  time=9.1s
[025/100] loss=0.5649  lr=1.79e-02  time=9.2s
[026/100] loss=0.5300  lr=1.77e-02

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
lr,▄▅▇██████████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁
time/epoch_sec,▂▇▂▄▅▁▆▅▅▂▃▁█▅▄▅▇▅▆▂▁▅▄▆▇▅▆▅▄▆▇▆▄▃▅▂▇▅▅▆
train/loss,█▇▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch,100
lr,0.0
time/epoch_sec,9.20939
train/loss,0.30994
